# Intro: A Minimal Local Eval Harness

This notebook shows the basic learning loop: load a small dataset, run a deterministic mock model, score the outputs, aggregate the results, and write a report.

In [ ]:
from pathlib import Path
import json

from eval_harness.aggregation import summarize_records
from eval_harness.dataset import load_jsonl_cases
from eval_harness.metrics import exact_match, normalized_exact_match, pass_fail_score
from eval_harness.reporting import write_json_report
from eval_harness.runner import run_cases

root = Path.cwd()
cases = load_jsonl_cases(root / "tests" / "evals" / "datasets" / "classification_cases.jsonl")
mock_outputs = json.loads((root / "tests" / "evals" / "mocks" / "mock_model_outputs.json").read_text())

def predict(case):
    return mock_outputs[case.case_id]

def score(case, actual):
    exact = pass_fail_score(exact_match(actual, case.expected))
    normalized = pass_fail_score(normalized_exact_match(actual, case.expected))
    passed = bool(normalized if case.metadata.get("match_type") == "normalized" else exact)
    return {"exact_match": exact, "normalized_exact_match": normalized}, passed

records = run_cases(cases, predict, scorer=score)
summary = summarize_records(records)
write_json_report(root / "outputs" / "notebook_intro_report.json", records, summary)
summary.model_dump()


## What You Will Learn

- How the repo loads a deterministic eval dataset.
- How a local predictor is run and scored.
- How records are aggregated into a summary report.
- Which files matter when you add or debug a simple eval case.

## Prerequisites

- The package is installed with `python3 -m pip install -e ".[dev]".
- The repo passes `make verify` before you start changing behavior.
- You are comfortable running one notebook cell at a time and reading pytest output.

## Key Repo Files
- `tests/evals/datasets/classification_cases.jsonl`
- `tests/evals/mocks/mock_model_outputs.json`
- `src/eval_harness/runner.py`
- `src/eval_harness/metrics.py`

The notebook shows the basic learning loop: load a small dataset, run a deterministic mock model, score the outputs, aggregate the results, and write a report.

In JupyterLab, do this:

1. Open notebooks/01_intro_harness.ipynb.
2. Click the first markdown cell.
3. Replace its contents with the block above.
4. Save with Cmd+S.

Then verify from a terminal:

```
cd $HOME/human-centered-eval-harnesses
make notebooks
```

Then make this a clean commit. You already have other local edits, so do not use git add .. Stage only the notebook:

```
git add notebooks/01_intro_harness.ipynb
git commit -m "docs: add learning goals to intro harness notebook"
git push
```

## Try it Yourself

Your first exercise is to add one new deterministic classification eval case.

### Goal

Add a case that teaches you how the harness and eval suite connect:
- the dataset defines the case
- the mock output defines what the model returned
- the eval behavior test decides whether that output should pass or fail

### Files You Will Edit

- `tests/evals/datasets/classification_cases.jsonl`
- `tests/evals/mocks/mock_model_outputs.json`

### Suggested Exercise

Add a near-miss case where the expected label is `"positive"` but the mock model output is `"positive-ish"`.

This should help you see why exact and normalized matching are intentionally strict for classification labels.

### After You Edit Those Files

Run:
```bash
PYTHONPATH=src python3 -m pytest tests/evals/test_eval_harness_behavior.py -v
make evals
```

### Reflection

Before you run tests, predict the results:

- Should this case pass or fail?
- Which field in metadata controls the expected outcome?
- Which part of the harness decides whether "positive-ish" counts as correct?

What to do in JupyterLab:

1. Open the notebook.
2. Click the `+` button to add a new cell above the code cell.
3. Change the cell type to `Markdown`.
4. Paste the block above.
5. Save with `Cmd+S`.

Then verify:

```
cd $HOME/human-centered-eval-harnesses
make notebooks

# Then commit only the notebook for this lesson checkpoint:
git add notebooks/01_intro_harness.ipynb
git commit -m "docs: add first exercise to intro harness notebook"
git push
```

Do not stage the dataset or mock files as part of commit 2. Those belong to the learner exercise or a later commit.

## Verify Your Change

After you add a new eval case and its mock output, do not guess whether it worked. Run the checks that prove it.

### Focused Check

Run the eval behavior test first:

```bash
PYTHONPATH=src python3 -m pytest tests/evals/test_eval_harness_behavior.py -v
```

This is the fastest way to confirm that your new case behaves the way its metadata says it should.

### Broader Check

Then run the full eval suite:

```bash
make evals
```

This confirms that :

- the eval behavior tests still pass
- the threshold local eval suite still passes

 ### Full Repo Check

 When the focused checks are green, run:

 ```bash
 make verify
 ```

 Use this only after the smaller checks pass.

 ### What To Look For

 Ask yourself:

 - Did the new case pass or fail as expected?
 - If it failed unexpectedly, was the issue in the dataset, the mock output, or the scoring logic?
 - Did a change in the dataset break any integration assumptions elsewhere?

Then:

1. Save the notebook.
2. Run:

```bash
cd $HOME/01_intro_harness.ipynb
make notebooks
```

3. Commit only the notebook:

```bash
git add notebooks/01_intro_harness.ipynb
git commit -m "docs: add verification workflow to intro notebook"
git push
```

Do not stage your dataset or mock changes in this commit. Those belong to a separate exercise or behavior commit.

## Reflect On The Result

After you run the tests, do not stop at green or red output. Explain why the result happened.

### Questions to Answer

- Which new cases did you add?
- What does its `metadata.expected_pass` value say should happen?
- Which file decides how classification outputs are scored?
- Why does `"positive-ish"` fail when the expected label is `"positive"`?
- Would you change the dataset, the mock output, or the metric if you wanted different behavior?

### Files to Inspect

- `tests/evals/test_eval_harness_behavior.py`
- `src/eval_harness/metrics.py`
- `tests/evals/datasets/classification_cases.jsonl`

### What You Should Learn

A passing test is not enough. You should be able to point to:
- the dataset row that defines the case
- the mock output that simulates the model behavior
- the scoring logic that decides pass or fail

Then do this:

1. Save the notebook.
2. Run:

```bash
cd $HOME/01_intro_harness.ipynb
make notebooks
```

3. Commit only the notebook:

```bash
git add notebooks/01_intro_harness.ipynb
git commit -m "docs: add reflection prompts for intro notebook"
git push
```

Do not stage dataset or mock changes in this commit. Keep commit 4 notebook-only.